# ============================================================
# Combined salary analysis (Adzuna + Job Bank)
# Strategy: keep both sources analytically distinct (juxtaposed)
# Geo granularity for Job Bank: Economic Region
# ============================================================

In [25]:
import pandas as pd
from pathlib import Path

#display settings

pd.set_option('display.max_columns',50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.0f}'.format)


# Path
PROCESSED_DIR = Path('../data/processed')

ADZUNA_PATH = PROCESSED_DIR/'adzuna_jobs_with_skills.csv'
JOBBANK_PATH = PROCESSED_DIR/'jobbank_ontario_data_clean.csv'

# Sanity check
for p in [ADZUNA_PATH, JOBBANK_PATH]:
    assert p.exists(), f"Missing file: {p.resolve()}"
# Load Adzuna
adzuna = pd.read_csv(ADZUNA_PATH)
#Load  Job Bank
jobbank = pd.read_csv(JOBBANK_PATH)
# Quick visual proof we loaded the rignt things
print(f"Adzuna  : {adzuna.shape[0]:>5} rows × {adzuna.shape[1]:>2} cols")
print(f"Job Bank: {jobbank.shape[0]:>5} rows × {jobbank.shape[1]:>2} cols")
print()
print("Adzuna columns:")
print(list(adzuna.columns))
print()
print("Job Bank columns:")
print(list(jobbank.columns))

Adzuna  :  1025 rows × 23 cols
Job Bank:  1111 rows × 21 cols

Adzuna columns:
['salary_max', 'latitude', 'redirect_url', 'title', 'created', 'salary_min', 'id', 'description', 'longitude', 'contract_time', '_search_query', 'company', 'category', 'category_tag', 'location', 'location_area', 'contract_type', 'city', 'week', 'day_of_week', 'is_recent', 'skills', 'text_to_search']

Job Bank columns:
['WIC Job Location Snapshot ID', 'First Posting Date', '_noc21_clean', 'Job Title', 'NOC21 Code Name', 'Province/Territory', 'City', 'Economic  Region', 'Salary Minimum', 'Salary Maximum', 'Salary Per', '_salary_per_corrected', 'salary_min_annual', 'salary_max_annual', 'salary_avg_annual', 'Education LOS', 'Experience Level', 'Hours Minimum', 'Hours Maximum', 'Hours Per', '_source_month']


In [26]:
#Diagnostic: Adzuna salary structure

# --- Fill rate ---
total = len(adzuna)
with_min = adzuna['salary_min'].notna().sum()
with_max = adzuna['salary_max'].notna().sum()
with_both = adzuna[['salary_min', 'salary_max']].notna().all(axis=1).sum()

print("ADZUNA SALARY FILL RATE")
print(f"  Total jobs            : {total:>5}")
print(f"  With salary_min       : {with_min:>5} ({with_min/total*100:.1f}%)")
print(f"  With salary_max       : {with_max:>5} ({with_max/total*100:.1f}%)")
print(f"  With both min and max : {with_both:>5} ({with_both/total*100:.1f}%)")
print()

# --- Distribution check: are these numbers in the annual CAD range? ---
print("ADZUNA SALARY DISTRIBUTION (raw values)")
print(adzuna[['salary_min', 'salary_max']].describe())
print()

# --- Sample of 5 random rows with salary, to eyeball the unit ---
print("ADZUNA SAMPLE (5 random jobs with salary)")
sample = adzuna[adzuna['salary_min'].notna()][
    ['title', '_search_query', 'salary_min', 'salary_max']
].sample(5, random_state=42)
print(sample.to_string(index=False))

ADZUNA SALARY FILL RATE
  Total jobs            :  1025
  With salary_min       :   258 (25.2%)
  With salary_max       :   258 (25.2%)
  With both min and max :   258 (25.2%)

ADZUNA SALARY DISTRIBUTION (raw values)
       salary_min  salary_max
count         258         258
mean       98,566     129,813
std        36,707      48,318
min        41,600      52,000
25%        72,000      95,000
50%        89,050     120,000
75%       120,000     150,000
max       228,000     313,500

ADZUNA SAMPLE (5 random jobs with salary)
                                    title   _search_query  salary_min  salary_max
                            Data Engineer    data analyst      90,000     125,000
Database Marketing Analyst (Data Science)    data analyst      72,400      90,500
                             Data Analyst    data analyst      87,000     108,500
Sales Representative- Microsoft Data & AI data consultant      50,000     120,000
Senior Manager, Clinical Decision Support  data scientist   

In [27]:
# ADZUNA : filter to jobs with salary, and salary_avg
adz = adzuna[adzuna['salary_min'].notna()].copy()

#recreate the salary avg

adz['_salary_avg'] = (adz['salary_min'] + adz['salary_max']) / 2


#Sanity check
assert len(adz) == 258 ,f"Expected 258 Adzuna jobs with salary, got {len(adz)}"

print("ADZUNA working dataframe")
print(f"  Rows : {len(adz)}")
print(f"  Cols : {len(adz.columns)} (added: _salary_avg)")
print(f"  _salary_avg median : {adz['_salary_avg'].median():,.0f} CAD/year")
print()

#job bank check of the dataset
jb = jobbank.copy()

print("JOB BANK working dataframe")
print(f"  Rows : {len(jb)}")
print(f"  Cols : {len(jb.columns)}")
print(f"  salary_avg_annual median : {jb['salary_avg_annual'].median():,.0f} CAD/year")
print()

#Headline 

total_with_salary = len(adz) + len(jb)
print(f"=> Combined salary-disclosed sample: {total_with_salary} jobs")
print(f"   ({len(adz)} Adzuna + {len(jb)} Job Bank)")



ADZUNA working dataframe
  Rows : 258
  Cols : 24 (added: _salary_avg)
  _salary_avg median : 105,000 CAD/year

JOB BANK working dataframe
  Rows : 1111
  Cols : 21
  salary_avg_annual median : 96,897 CAD/year

=> Combined salary-disclosed sample: 1369 jobs
   (258 Adzuna + 1111 Job Bank)


# ============================================================
#  Salary by ROLE, Adzuna source
# Taxonomy: _search_query (the keyword used to query the API)
# Method: median 
# Threshold: n >= 5 to be publishable
# ============================================================

In [28]:
MIN_SAMPLE_SIZE = 5

salary_by_role_adzuna = (
   adz
    .groupby('_search_query')['_salary_avg']
    .agg(['count', 'median', 'mean', 'min', 'max'])
    .round(0)
    .sort_values('median', ascending=False)
)
# Apply publishability filter
salary_by_role_adzuna_pub = salary_by_role_adzuna[
    salary_by_role_adzuna['count'] >= MIN_SAMPLE_SIZE
]


print(f"ADZUNA — Salary by role")
print(f"  Roles total       : {len(salary_by_role_adzuna)}")
print(f"  Roles publishable : {len(salary_by_role_adzuna_pub)} (n >= {MIN_SAMPLE_SIZE})")
print()

# Show the full table including small-n roles, but flag them
print("FULL TABLE (with sample size):")
print(salary_by_role_adzuna)
print()

print("PUBLISHABLE TABLE (n >= 5):")
print(salary_by_role_adzuna_pub)

ADZUNA — Salary by role
  Roles total       : 6
  Roles publishable : 5 (n >= 5)

FULL TABLE (with sample size):
                  count  median    mean    min     max
_search_query                                         
data consultant      35 137,500 141,785 52,500 232,000
data scientist       44 130,000 129,393 69,582 215,000
data engineer        47 120,000 124,577 55,120 270,750
BI analyst            4 107,400 103,620 74,880 124,800
business analyst     60  92,446 103,015 55,000 228,800
data analyst         68  89,938  93,452 46,800 209,500

PUBLISHABLE TABLE (n >= 5):
                  count  median    mean    min     max
_search_query                                         
data consultant      35 137,500 141,785 52,500 232,000
data scientist       44 130,000 129,393 69,582 215,000
data engineer        47 120,000 124,577 55,120 270,750
business analyst     60  92,446 103,015 55,000 228,800
data analyst         68  89,938  93,452 46,800 209,500


# ============================================================
#  Salary by ROLE, Job Bank source
# Taxonomy: _search_query (the keyword used to query the API)
# Method: median 
# Threshold: n >= 5 to be publishable
# ============================================================

In [29]:
salary_by_role_jobbank = (
    jb
    .groupby(['_noc21_clean', 'NOC21 Code Name'])['salary_avg_annual']
    .agg(['count', 'median', 'mean', 'min', 'max'])
    .round(0)
    .sort_values('median', ascending=False)
)

# Same publishability filter (consistency across the analysis)
salary_by_role_jobbank_pub = salary_by_role_jobbank[
    salary_by_role_jobbank['count'] >= MIN_SAMPLE_SIZE
]

print(f"JOB BANK — Salary by role (NOC 2021)")
print(f"  Roles total       : {len(salary_by_role_jobbank)}")
print(f"  Roles publishable : {len(salary_by_role_jobbank_pub)} (n >= {MIN_SAMPLE_SIZE})")
print()

print("FULL TABLE:")
print(salary_by_role_jobbank)
print()

print("PUBLISHABLE TABLE (n >= 5):")
print(salary_by_role_jobbank_pub)

JOB BANK — Salary by role (NOC 2021)
  Roles total       : 4
  Roles publishable : 4 (n >= 5)

FULL TABLE:
                                                        count  median    mean    min     max
_noc21_clean NOC21 Code Name                                                                
21211        Data scientists                              215 103,730 104,587 32,500 197,600
21221        Business systems specialists                 175  96,897  96,773 45,760 176,800
21223        Database analysts and data administrators     72  89,471  89,531 38,688 160,000
21222        Information systems specialists              649  85,280  87,100 26,260 205,442

PUBLISHABLE TABLE (n >= 5):
                                                        count  median    mean    min     max
_noc21_clean NOC21 Code Name                                                                
21211        Data scientists                              215 103,730 104,587 32,500 197,600
21221        Business syste

# ============================================================
# Side-by-side view: salary by role across both sources
# Strategy: NO mapping, NO merge — both tables stand on their own
# Goal: reader sees both market lenses + their structural difference
# ============================================================

In [30]:
print("=" * 70)
print("ONTARIO DATA SALARIES BY ROLE — TWO MARKET LENSES")
print("=" * 70)
print()
print(f"{'ADZUNA (private-sector aggregator)':<70}")
print(f"{'Source: Adzuna API · Sample: ' + str(len(adz)) + ' jobs · Median: ' + format(adz['_salary_avg'].median(), ',.0f') + ' CAD':<70}")
print("-" * 70)
print(salary_by_role_adzuna_pub.to_string())
print()
print(f"{'JOB BANK (Open Government Licence Canada)':<70}")
print(f"{'Source: Job Bank ESDC · Sample: ' + str(len(jb)) + ' jobs · Median: ' + format(jb['salary_avg_annual'].median(), ',.0f') + ' CAD':<70}")
print("-" * 70)
print(salary_by_role_jobbank_pub.to_string())
print()
print("=" * 70)
print("STRUCTURAL OBSERVATIONS")
print("=" * 70)

# Structural metrics that highlight WHY we kept them separate
adz_range = (
    salary_by_role_adzuna_pub['median'].max()
    - salary_by_role_adzuna_pub['median'].min()
)
jb_range = (
    salary_by_role_jobbank_pub['median'].max()
    - salary_by_role_jobbank_pub['median'].min()
)

print(f"  Median spread Adzuna   : {adz_range:>7,.0f} CAD (top - bottom role)")
print(f"  Median spread Job Bank : {jb_range:>7,.0f} CAD (top - bottom role)")
print(f"  => Adzuna shows {adz_range/jb_range:.1f}x the role-to-role differentiation")
print()

adz_top = salary_by_role_adzuna_pub.index[0]
jb_top = salary_by_role_jobbank_pub.index[0][1]  # [1] = the human label, not the code
print(f"  Top-paying role (Adzuna)   : {adz_top}")
print(f"  Top-paying role (Job Bank) : {jb_top}")

ONTARIO DATA SALARIES BY ROLE — TWO MARKET LENSES

ADZUNA (private-sector aggregator)                                    
Source: Adzuna API · Sample: 258 jobs · Median: 105,000 CAD           
----------------------------------------------------------------------
                  count  median    mean    min     max
_search_query                                         
data consultant      35 137,500 141,785 52,500 232,000
data scientist       44 130,000 129,393 69,582 215,000
data engineer        47 120,000 124,577 55,120 270,750
business analyst     60  92,446 103,015 55,000 228,800
data analyst         68  89,938  93,452 46,800 209,500

JOB BANK (Open Government Licence Canada)                             
Source: Job Bank ESDC · Sample: 1111 jobs · Median: 96,897 CAD        
----------------------------------------------------------------------
                                                        count  median    mean    min     max
_noc21_clean NOC21 Code Name                

# ============================================================
# Salary by CITY, Adzuna source
# Diagnostic on the city column before any aggregation
# Build salary_by_city_adzuna with same threshold logic
# ============================================================

In [31]:
print("ADZUNA city column — diagnostic")
print(f"  Unique cities (with salary)        : {adz['city'].nunique()}")
print(f"  Top 10 by job count:")
print(adz['city'].value_counts().head(10).to_string())
print()
print(f"  Missing/null cities                 : {adz['city'].isna().sum()}")

ADZUNA city column — diagnostic
  Unique cities (with salary)        : 17
  Top 10 by job count:
city
Toronto            151
Peel region         27
Ottawa region       17
York region         13
Halton               9
Waterloo region      9
Middlesex            6
Durham region        5
Hamilton region      3
Elgin region         2

  Missing/null cities                 : 8


In [32]:
# ============================================================
# Salary by CITY, Adzuna (with bucketing)
# Strategy: keep all n>=5 zones individual + Other + Unknown
# ============================================================

# identify which cities meet the threshold ---
city_counts = adz['city'].value_counts()
publishable_cities = city_counts[city_counts >= MIN_SAMPLE_SIZE].index.tolist()

print(f"Publishable cities (n >= {MIN_SAMPLE_SIZE}): {len(publishable_cities)}")
print(publishable_cities)
print()

# create a bucketed city column ---
# Convention: raw 'city' stays sacred, '_city_bucket' is the engineered one
def bucket_city(c):
    if pd.isna(c):
        return 'Unknown'
    elif c in publishable_cities:
        return c
    else:
        return 'Other Ontario'

adz['_city_bucket'] = adz['city'].apply(bucket_city)

# --- Step 3: aggregate ---
salary_by_city_adzuna = (
    adz
    .groupby('_city_bucket')['_salary_avg']
    .agg(['count', 'median', 'mean', 'min', 'max'])
    .round(0)
    .sort_values('median', ascending=False)
)

print("ADZUNA — Salary by city/region")
print(salary_by_city_adzuna.to_string())
print()

# --- Sanity check: the buckets should sum to 258 ---
total_check = salary_by_city_adzuna['count'].sum()
assert total_check == 258, f"Bucket counts don't sum to 258: {total_check}"
print(f"✅ Total check: {total_check} jobs (= Adzuna salary sample)")

Publishable cities (n >= 5): 8
['Toronto', 'Peel region', 'Ottawa region', 'York region', 'Halton', 'Waterloo region', 'Middlesex', 'Durham region']

ADZUNA — Salary by city/region
                 count  median    mean     min     max
_city_bucket                                          
Durham region        5 176,800 173,056 135,200 199,680
Unknown              8 119,500 138,187  78,000 204,000
Ottawa region       17 112,650 121,062  46,800 225,000
Middlesex            6 110,125 131,625  73,000 208,000
Toronto            151 108,000 117,115  52,500 270,750
Waterloo region      9  98,500 103,886  57,148 154,570
Halton               9  97,129 101,959  75,000 145,400
York region         13  89,877  89,513  55,000 120,000
Peel region         27  89,702 101,546  65,000 225,000
Other Ontario       13  78,500  92,315  48,000 225,000

✅ Total check: 258 jobs (= Adzuna salary sample)


# ============================================================
# Salary by CITY, Job Bank source
# Diagnostic on the city column before any aggregation
# Build salary_by_city_adzuna with same threshold logic
# ============================================================

In [33]:
ECON_REGION = 'Economic  Region'

print("JOB BANK Economic Region — diagnostic")
print(f"  Unique regions          : {jb[ECON_REGION].nunique()}")
print(f"  Missing/null            : {jb[ECON_REGION].isna().sum()}")
print()
print("  Full distribution by job count:")
print(jb[ECON_REGION].value_counts(dropna=False).to_string())

JOB BANK Economic Region — diagnostic
  Unique regions          : 11
  Missing/null            : 10

  Full distribution by job count:
Economic  Region
Toronto Region                       946
Kitchener–Waterloo–Barrie Region      35
Ottawa Region                         31
Hamilton–Niagara Peninsula Region     26
Northeast Region                      22
London Region                         21
Windsor-Sarnia Region                 11
NaN                                   10
Kingston–Pembroke Region               4
Stratford–Bruce Peninsula Region       2
Northwest Region                       2
Muskoka–Kawarthas Region               1


In [34]:
# ============================================================
#Salary by ECONOMIC REGION, Job Bank (with bucketing)
# Same bucketing pattern as Cell 8 for cross-source consistency
# ============================================================

ECON_REGION = 'Economic  Region'  # /!\ TWO spaces

# --- Step 1: identify publishable regions ---
region_counts = jb[ECON_REGION].value_counts()
publishable_regions = region_counts[region_counts >= MIN_SAMPLE_SIZE].index.tolist()

print(f"Publishable regions (n >= {MIN_SAMPLE_SIZE}): {len(publishable_regions)}")
for r in publishable_regions:
    print(f"  - {r} ({region_counts[r]})")
print()

# --- Step 2: bucket the region column ---
def bucket_region(r):
    if pd.isna(r):
        return 'Unknown'
    elif r in publishable_regions:
        return r
    else:
        return 'Other Ontario'

jb['_region_bucket'] = jb[ECON_REGION].apply(bucket_region)

# --- Step 3: aggregate ---
salary_by_region_jobbank = (
    jb
    .groupby('_region_bucket')['salary_avg_annual']
    .agg(['count', 'median', 'mean', 'min', 'max'])
    .round(0)
    .sort_values('median', ascending=False)
)

print("JOB BANK — Salary by Economic Region")
print(salary_by_region_jobbank.to_string())
print()

# --- Sanity check ---
total_check = salary_by_region_jobbank['count'].sum()
assert total_check == len(jb), f"Bucket counts don't sum to {len(jb)}: {total_check}"
print(f"✅ Total check: {total_check} jobs (= Job Bank salary sample)")

Publishable regions (n >= 5): 7
  - Toronto Region (946)
  - Kitchener–Waterloo–Barrie Region (35)
  - Ottawa Region (31)
  - Hamilton–Niagara Peninsula Region (26)
  - Northeast Region (22)
  - London Region (21)
  - Windsor-Sarnia Region (11)

JOB BANK — Salary by Economic Region
                                   count  median   mean    min     max
_region_bucket                                                        
London Region                         21  96,897 84,163 45,011 118,718
Toronto Region                       946  96,897 95,358 26,260 205,442
Ottawa Region                         31  90,860 88,502 41,600 170,000
Unknown                               10  72,900 77,100 33,000 166,400
Windsor-Sarnia Region                 11  65,603 65,305 41,600  85,000
Kitchener–Waterloo–Barrie Region      35  62,400 68,937 38,480 130,000
Other Ontario                          9  62,400 67,479 36,254 110,000
Northeast Region                      22  62,400 69,137 48,194 137,500
Hamilto

In [35]:
# ============================================================
#  Side-by-side view: salary by region across both sources
# Same juxtaposition strategy as Cell 6 (no merge, no mapping)
# ============================================================

print("=" * 70)
print("ONTARIO DATA SALARIES BY REGION — TWO MARKET LENSES")
print("=" * 70)
print()
print("ADZUNA (private-sector aggregator)")
print(f"Source: Adzuna API · Sample: {len(adz)} jobs · Median: {adz['_salary_avg'].median():,.0f} CAD")
print("-" * 70)
print(salary_by_city_adzuna.to_string())
print()
print("JOB BANK (Open Government Licence Canada)")
print(f"Source: Job Bank ESDC · Sample: {len(jb)} jobs · Median: {jb['salary_avg_annual'].median():,.0f} CAD")
print("-" * 70)
print(salary_by_region_jobbank.to_string())
print()
print("=" * 70)
print("STRUCTURAL OBSERVATIONS")
print("=" * 70)

# --- Geographic concentration metrics ---
adz_top_share = (
    salary_by_city_adzuna.loc['Toronto', 'count']
    / salary_by_city_adzuna['count'].sum() * 100
)
jb_top_share = (
    salary_by_region_jobbank.loc['Toronto Region', 'count']
    / salary_by_region_jobbank['count'].sum() * 100
)

print(f"  Toronto/Toronto Region share — Adzuna   : {adz_top_share:.1f}%")
print(f"  Toronto/Toronto Region share — Job Bank : {jb_top_share:.1f}%")
print(f"  /!\\ NOT directly comparable: Adzuna 'Toronto' = city only,")
print(f"      Job Bank 'Toronto Region' = full GTA (incl. Peel/York/Halton/Durham)")
print()

# --- Range of medians ---
adz_region_range = (
    salary_by_city_adzuna['median'].max()
    - salary_by_city_adzuna['median'].min()
)
jb_region_range = (
    salary_by_region_jobbank['median'].max()
    - salary_by_region_jobbank['median'].min()
)
print(f"  Median spread Adzuna   : {adz_region_range:>7,.0f} CAD (top - bottom region)")
print(f"  Median spread Job Bank : {jb_region_range:>7,.0f} CAD (top - bottom region)")
print()

# --- Hypothesis test: Ottawa share ---
adz_ottawa = (
    salary_by_city_adzuna.loc['Ottawa region', 'count']
    / salary_by_city_adzuna['count'].sum() * 100
)
jb_ottawa = (
    salary_by_region_jobbank.loc['Ottawa Region', 'count']
    / salary_by_region_jobbank['count'].sum() * 100
)
print(f"  Ottawa share — Adzuna   : {adz_ottawa:.1f}%")
print(f"  Ottawa share — Job Bank : {jb_ottawa:.1f}%")
print(f"  => Initial Session 4 hypothesis 'Job Bank captures more Ottawa")
print(f"     fed-gov jobs' is INVALIDATED by data ({jb_ottawa:.1f}% < {adz_ottawa:.1f}%)")

ONTARIO DATA SALARIES BY REGION — TWO MARKET LENSES

ADZUNA (private-sector aggregator)
Source: Adzuna API · Sample: 258 jobs · Median: 105,000 CAD
----------------------------------------------------------------------
                 count  median    mean     min     max
_city_bucket                                          
Durham region        5 176,800 173,056 135,200 199,680
Unknown              8 119,500 138,187  78,000 204,000
Ottawa region       17 112,650 121,062  46,800 225,000
Middlesex            6 110,125 131,625  73,000 208,000
Toronto            151 108,000 117,115  52,500 270,750
Waterloo region      9  98,500 103,886  57,148 154,570
Halton               9  97,129 101,959  75,000 145,400
York region         13  89,877  89,513  55,000 120,000
Peel region         27  89,702 101,546  65,000 225,000
Other Ontario       13  78,500  92,315  48,000 225,000

JOB BANK (Open Government Licence Canada)
Source: Job Bank ESDC · Sample: 1111 jobs · Median: 96,897 CAD
---------------

In [36]:
# ============================================================
#Salary by SKILL, Adzuna only
# Why Adzuna only: Job Bank has no free-text description column
# ============================================================

import ast  # to safely parse the stringified list back into a Python list

# --- Step 1: filter to jobs that have BOTH salary AND skills ---
mask_salary = adz['_salary_avg'].notna()
mask_skills = adz['skills'].notna() & (adz['skills'] != '[]')

adz_skill_sal = adz[mask_salary & mask_skills].copy()
print(f"Adzuna jobs with BOTH salary AND skills: {len(adz_skill_sal)}")
print()

# --- Step 2: parse the stringified list ---
# In CSV, a Python list ['SQL', 'Python'] is saved as the string "['SQL', 'Python']"
# ast.literal_eval safely converts it back into an actual list
adz_skill_sal['_skills_list'] = adz_skill_sal['skills'].apply(ast.literal_eval)

# Quick sanity check: are these really lists now?
print("Sample of parsed skills (first 3 jobs):")
for skills_list in adz_skill_sal['_skills_list'].head(3):
    print(f"  {type(skills_list).__name__}: {skills_list}")
print()

# --- Step 3: explode (one row per skill per job) ---
# A job with skills ['SQL', 'Python', 'PowerBI'] becomes 3 rows
adz_exploded = adz_skill_sal.explode('_skills_list')
print(f"After explode: {len(adz_exploded)} rows (skill-job pairs)")
print(f"  Unique skills observed: {adz_exploded['_skills_list'].nunique()}")
print()

# --- Step 4: aggregate ---
salary_by_skill = (
    adz_exploded
    .groupby('_skills_list')['_salary_avg']
    .agg(['count', 'median', 'mean', 'min', 'max'])
    .round(0)
    .sort_values('median', ascending=False)
)

salary_by_skill_pub = salary_by_skill[salary_by_skill['count'] >= MIN_SAMPLE_SIZE]

print(f"ADZUNA — Salary by skill")
print(f"  Skills total       : {len(salary_by_skill)}")
print(f"  Skills publishable : {len(salary_by_skill_pub)} (n >= {MIN_SAMPLE_SIZE})")
print()
print("PUBLISHABLE TABLE (n >= 5):")
print(salary_by_skill_pub.to_string())

Adzuna jobs with BOTH salary AND skills: 61

Sample of parsed skills (first 3 jobs):
  list: ['Dashboards']
  list: ['Dashboards', 'Reporting']
  list: ['Python', 'SQL', 'Excel', 'Statistics']

After explode: 99 rows (skill-job pairs)
  Unique skills observed: 23

ADZUNA — Salary by skill
  Skills total       : 23
  Skills publishable : 7 (n >= 5)

PUBLISHABLE TABLE (n >= 5):
                     count  median    mean     min     max
_skills_list                                              
Azure                    8 127,400 134,025 105,000 199,680
SQL                      6 121,680 113,100  89,440 128,500
Data Pipeline            9 120,000 112,039  55,120 142,480
Machine Learning         9  95,700 127,289  72,500 270,750
Dashboards               7  92,393  93,964  69,582 125,000
Reporting               23  89,877  91,202  55,000 147,600
Predictive Modeling      5  72,500  83,640  65,000 112,500


In [37]:
# ============================================================
# Synthesis + save outputs for Session 7 (visualizations)
# Pattern: each output is a CSV in outputs/, named consistently
# Convention: <axis>_<source>_combined.csv when 2 sources juxtaposed
# ============================================================

OUTPUTS_DIR = Path('../outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)  # creates if missing, no-op otherwise

# ------------------------------------------------------------
# Save each table with a clear naming convention
# ------------------------------------------------------------

# Axis 1 — by role (2 sources, separate files for clarity)
salary_by_role_adzuna_pub.to_csv(
    OUTPUTS_DIR / 'salary_by_role_adzuna.csv', encoding='utf-8-sig'
)
salary_by_role_jobbank_pub.to_csv(
    OUTPUTS_DIR / 'salary_by_role_jobbank.csv', encoding='utf-8-sig'
)

# Axis 2 — by region (2 sources, separate files)
salary_by_city_adzuna.to_csv(
    OUTPUTS_DIR / 'salary_by_region_adzuna.csv', encoding='utf-8-sig'
)
salary_by_region_jobbank.to_csv(
    OUTPUTS_DIR / 'salary_by_region_jobbank.csv', encoding='utf-8-sig'
)

# Axis 3 — by skill (Adzuna only, by design)
salary_by_skill_pub.to_csv(
    OUTPUTS_DIR / 'salary_by_skill_adzuna.csv', encoding='utf-8-sig'
)

print("✅ Outputs saved to /outputs/")
for f in sorted(OUTPUTS_DIR.glob('salary_by_*')):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name} ({size_kb:.1f} KB)")
print()

# ------------------------------------------------------------
# Headline numbers for the README
# ------------------------------------------------------------
print("=" * 70)
print("SESSION 6 — KEY NUMBERS FOR README")
print("=" * 70)
print(f"  Combined salary-disclosed sample : 1,369 jobs")
print(f"  Adzuna jobs with salary           : {len(adz)}")
print(f"  Job Bank jobs with salary         : {len(jb)}")
print(f"  Skill × salary sample (Adzuna)    : {len(adz_skill_sal)}")
print()
print(f"  TOP-PAYING ROLE")
print(f"    Adzuna   : data consultant (median 137,500 CAD, n=35)")
print(f"    Job Bank : Data scientists NOC 21211 (median 103,730 CAD, n=215)")
print()
print(f"  TOP-PAYING REGION")
print(f"    Adzuna   : Durham region (median 176,800 CAD, n=5 — small sample caveat)")
print(f"    Job Bank : Toronto Region & London Region tied (median 96,897 CAD)")
print()
print(f"  TOP-PAYING SKILL (Adzuna only)")
print(f"    Azure (median 127,400 CAD, n=8)")
print()
print(f"  STRUCTURAL FINDINGS")
print(f"    - Adzuna shows 2.6x role differentiation vs Job Bank")
print(f"    - Adzuna shows 2.4x region differentiation vs Job Bank")
print(f"    - Hypothesis 'Job Bank captures more Ottawa fed-gov jobs': INVALIDATED")
print(f"    - Median Job Bank ($96,897) recurs 3x: global, NOC 21221, Toronto Region")
print(f"      => suggests anchored salary grid in public/contractual segment")

✅ Outputs saved to /outputs/
   salary_by_city.csv (0.3 KB)
   salary_by_region_adzuna.csv (0.5 KB)
   salary_by_region_jobbank.csv (0.5 KB)
   salary_by_role.csv (0.3 KB)
   salary_by_role_adzuna.csv (0.3 KB)
   salary_by_role_jobbank.csv (0.3 KB)
   salary_by_skill.csv (0.4 KB)
   salary_by_skill_adzuna.csv (0.4 KB)

SESSION 6 — KEY NUMBERS FOR README
  Combined salary-disclosed sample : 1,369 jobs
  Adzuna jobs with salary           : 258
  Job Bank jobs with salary         : 1111
  Skill × salary sample (Adzuna)    : 61

  TOP-PAYING ROLE
    Adzuna   : data consultant (median 137,500 CAD, n=35)
    Job Bank : Data scientists NOC 21211 (median 103,730 CAD, n=215)

  TOP-PAYING REGION
    Adzuna   : Durham region (median 176,800 CAD, n=5 — small sample caveat)
    Job Bank : Toronto Region & London Region tied (median 96,897 CAD)

  TOP-PAYING SKILL (Adzuna only)
    Azure (median 127,400 CAD, n=8)

  STRUCTURAL FINDINGS
    - Adzuna shows 2.6x role differentiation vs Job Bank
    -

In [38]:
# Files made obsolete by Session 6 (replaced by suffixed versions)
OBSOLETE_FILES = [
    'salary_by_role.csv',
    'salary_by_city.csv',
    'salary_by_skill.csv',
]

for fname in OBSOLETE_FILES:
    f = OUTPUTS_DIR / fname
    if f.exists():
        f.unlink()  # delete the file
        print(f"  🗑️  Removed {fname}")
    else:
        print(f"  ⚠️  Not found (already gone?): {fname}")

print()
print("Final state of /outputs/:")
for f in sorted(OUTPUTS_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.1f} KB)")

  🗑️  Removed salary_by_role.csv
  🗑️  Removed salary_by_city.csv
  🗑️  Removed salary_by_skill.csv

Final state of /outputs/:
  salary_by_region_adzuna.csv (0.5 KB)
  salary_by_region_jobbank.csv (0.5 KB)
  salary_by_role_adzuna.csv (0.3 KB)
  salary_by_role_jobbank.csv (0.3 KB)
  salary_by_skill_adzuna.csv (0.4 KB)
  skills_by_job_category.csv (0.8 KB)
  top_skills_overall.csv (0.3 KB)
